In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time 


In [ ]:
base_url = 'https://www.moteur.ma/fr/neuf/fiche-technique-recherche/?budget_min=100&budget_max=10000000000&marque=0&modele=0&carburant=0&boite=0&carross&per_page='
datas = []
# Loop through multiple pages
for page in range(1, 1000, 10):  # Adjust the range for more pages
    print(f"Scraping page {page}...")
    url = base_url + str(page)
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        page_content = response.text
    else:
        print(f'Failed to retrieve the page. Status code: {response.status_code}')    

    soup = BeautifulSoup(page_content, 'html.parser')
    car_models = soup.find_all('div', class_='col-md-12 white')  # Replace with actual class
    
    if not car_models:
        print("No more car listings found. Stopping.")
        break
    for model in car_models:
        model_Name = model.find('h2', style="font-size:19px;margin-top: 12px;")
        price_Intervall = model.find('h3', style="margin-top: 0;")
        model_Name = model_Name.text.strip() if model_Name else "N/A"
        price_Intervall = price_Intervall.text.strip() if price_Intervall else "N/A"
        car_versions = model.find_all('div', class_='col-md-12 col-sm-12 col-lg-12 col-xs-12 no-padding only_mobile list-new-mobile')
        for version in car_versions:
            Version = version.find('span', style="font-weight:bold")
            Version_price = version.find('span', style="color:#f04c24;")
            Version = Version.text.strip() if Version else "N/A"
            Version_price = Version_price.text.strip() if Version_price else "N/A"
            datas.append({'MODEL': model_Name, 'INTERVALE': price_Intervall, 'VERSION': Version, 'VERSION_PRICE': Version_price})
    time.sleep(2)  # Add delay to prevent blocking
for data in datas:
    print(data)


In [ ]:
df = pd.DataFrame(datas)
df.to_csv('car_listings.csv', index=False)
